In [1]:
import sys
sys.path.append("../src")
from data_loader import (load_prices, compute_hedge_ratio,
                          compute_spread, compute_zscore,
                          generate_signals, compute_pnl_with_costs)

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

prices = load_prices()

beta_MA_V, intercept_MA_V = compute_hedge_ratio(prices, "MA", "V")
beta_GS_MS, intercept_GS_MS = compute_hedge_ratio(prices, "GS", "MS")

spread_MA_V = compute_spread(prices, "MA", "V", beta_MA_V, intercept_MA_V)
spread_GS_MS = compute_spread(prices, "GS", "MS", beta_GS_MS, intercept_GS_MS)

zscore_MA_V = compute_zscore(spread_MA_V)
zscore_GS_MS = compute_zscore(spread_GS_MS)

signals_MA_V = generate_signals(zscore_MA_V)
signals_GS_MS = generate_signals(zscore_GS_MS)

pnl_MA_V = compute_pnl_with_costs(prices, "MA", "V", beta_MA_V, signals_MA_V)
pnl_GS_MS = compute_pnl_with_costs(prices, "GS", "MS", beta_GS_MS, signals_GS_MS)

portfolio_pnl = pnl_MA_V + pnl_GS_MS

print("P&L loaded.")
print(f"Portfolio total: ${portfolio_pnl.sum():.2f}")

P&L loaded.
Portfolio total: $2224.50


In [2]:
def sharpe_ratio(pnl , period_per_year = 252):
    daily_returns = pnl / 10000
    mean_return = daily_returns.mean()
    std_return = daily_returns.std()

    if std_return == 0:
        return 0
    
    sharpe = (mean_return / std_return) * np.sqrt(period_per_year)
    return round(sharpe,4)

print("Sharpe Ratio : ")
print(f"  MA/V   : {sharpe_ratio(pnl_MA_V)}")
print(f"  GS/MA  : {sharpe_ratio(pnl_GS_MS)}")
print(f"  Portfolio : {sharpe_ratio(portfolio_pnl)}")

Sharpe Ratio : 
  MA/V   : 0.414
  GS/MA  : 0.3431
  Portfolio : 0.5225


In [3]:
def max_drawdown(pnl):
    cumulaltive = pnl.cumsum()
    rolling_peak = cumulaltive.cummax()
    drawdown = (cumulaltive - rolling_peak) / (rolling_peak + 10000)
    max_dd = drawdown.min()
    return round(max_dd * 100 , 2)

print("Maximum Drawdown : ")
print(f"  MA/V  : {max_drawdown(pnl_MA_V)} %")
print(f"  GS/MS : {max_drawdown(pnl_GS_MS)} %")
print(f"  Portfolio : {max_drawdown(portfolio_pnl)} %")

Maximum Drawdown : 
  MA/V  : -7.5 %
  GS/MS : -7.96 %
  Portfolio : -8.55 %


In [14]:
def trade_statistics(pnl, signals , pair_name , capital = 10000):
    signal_shifted = signals.shift(1).fillna(0)

    trades = []
    in_trade = False
    trade_pnl = 0
    trade_days = 0

    for i in range(len(pnl)):
        sig = float(signal_shifted.iloc[i])
        daily = float(pnl.iloc[i])

        if sig != 0.0  and not in_trade:
            in_trade = True
            trade_pnl = daily 
            trade_days = 1

        elif sig != 0.0 and in_trade:
            trade_pnl += daily
            trade_days += 1
        
        elif sig == 0.0 and in_trade:
            trades.append({
                "pnl" : round(trade_pnl,2),
                "duration_days" : trade_days,
                "win" : trade_pnl  > 0
            })

            in_trade = False
            trade_pnl = 0
            trade_days = 0

    trades_df = pd.DataFrame(trades)

    win_rate = trades_df["win"].mean() * 100
    avg_win = trades_df[trades_df["win"]]["pnl"].mean()
    avg_loss = trades_df[~trades_df["win"]]["pnl"].mean()
    avg_duration = trades_df["duration_days"].mean()

    print(f"{pair_name} Trade Statistics : ")
    print(f"  Total trades  : {len(trades_df)}")
    print(f"  Win rate         : {win_rate:.1f}%")
    print(f"  Avg winning trade: ${avg_win:.2f}")
    print(f"  Avg losing trade : ${avg_loss:.2f}")
    print(f"  Avg duration     : {avg_duration:.1f} days")
    print(f"  Total P&L        : ${trades_df['pnl'].sum():.2f}")
    print()
    
    return trades_df

trades_MA_V = trade_statistics(pnl_MA_V, signals_MA_V, "MA/V")
trades_GS_MS = trade_statistics(pnl_GS_MS, signals_GS_MS, "GS/MS")


MA/V Trade Statistics : 
  Total trades  : 33
  Win rate         : 81.8%
  Avg winning trade: $88.19
  Avg losing trade : $-144.44
  Avg duration     : 18.3 days
  Total P&L        : $1514.42

GS/MS Trade Statistics : 
  Total trades  : 29
  Win rate         : 69.0%
  Avg winning trade: $158.35
  Avg losing trade : $-166.10
  Avg duration     : 20.4 days
  Total P&L        : $1672.09



In [10]:
signal_shifted = signals_MA_V.shift(1).fillna(0)

print("Signal value counts:")
print(signal_shifted.value_counts())

print("\nFirst 10 transitions:")
for i in range(1, 20):
    prev = signal_shifted.iloc[i-1]
    curr = signal_shifted.iloc[i]
    if prev != curr:
        print(f"  Day {i}: {prev} → {curr}")

Signal value counts:
 0.0    905
 1.0    326
-1.0    278
Name: count, dtype: int64

First 10 transitions:


In [11]:
print(type(signal_shifted.iloc[0]))
print(signal_shifted.iloc[0] == 0)
print(signal_shifted.iloc[0] != 0)

# check where transitions actually happen
changes = (signal_shifted != signal_shifted.shift(1))
print("Number of transitions found:", changes.sum())
print(signal_shifted[changes].head(10))

<class 'numpy.float64'>
True
False
Number of transitions found: 67
Date
2018-01-02    0.0
2018-04-27    1.0
2018-05-03    0.0
2018-07-27    1.0
2018-09-06    0.0
2018-11-02    1.0
2018-12-06    0.0
2018-12-26   -1.0
2018-12-28    0.0
2019-01-29   -1.0
dtype: float64


In [16]:
def metrics_report(pnl , signals , pair_name , capital = 10000):
    trade_df = trade_statistics(pnl , signals , pair_name)

    sharpe = sharpe_ratio(pnl)
    max_dd = max_drawdown(pnl)
    total_return = round((pnl.sum() / capital) * 100, 2)

    print(f"{'='*40}")
    print(f" {pair_name} = Full Metrics Report")
    print(f"{'='*40}")
    print(f"  Total return : {total_return} %")
    print(f" Sharpe Ratio : {sharpe}")
    print(f"  Max drawdown  : {max_dd} %")
    print(f"  Total trades  : {len(trade_df)}")
    print(f" Win rate  : {round(trade_df['win'].mean()*100,1)}%")
    print(f" Avg duration  : {round(trade_df['duration_days'].mean(),1)} days")
    print(f"{'=' *40}")
    print()

metrics_report(pnl_MA_V , signals_MA_V , "MA/V")
metrics_report(pnl_GS_MS , signals_GS_MS , "GS/MS")
metrics_report(portfolio_pnl , signals_MA_V , "Portfolio")

MA/V Trade Statistics : 
  Total trades  : 33
  Win rate         : 81.8%
  Avg winning trade: $88.19
  Avg losing trade : $-144.44
  Avg duration     : 18.3 days
  Total P&L        : $1514.42

 MA/V = Full Metrics Report
  Total return : 10.19 %
 Sharpe Ratio : 0.414
  Max drawdown  : -7.5 %
  Total trades  : 33
 Win rate  : 81.8%
 Avg duration  : 18.3 days

GS/MS Trade Statistics : 
  Total trades  : 29
  Win rate         : 69.0%
  Avg winning trade: $158.35
  Avg losing trade : $-166.10
  Avg duration     : 20.4 days
  Total P&L        : $1672.09

 GS/MS = Full Metrics Report
  Total return : 12.05 %
 Sharpe Ratio : 0.3431
  Max drawdown  : -7.96 %
  Total trades  : 29
 Win rate  : 69.0%
 Avg duration  : 20.4 days

Portfolio Trade Statistics : 
  Total trades  : 33
  Win rate         : 69.7%
  Avg winning trade: $106.46
  Avg losing trade : $-111.59
  Avg duration     : 18.3 days
  Total P&L        : $1332.66

 Portfolio = Full Metrics Report
  Total return : 22.24 %
 Sharpe Ratio : 

In [18]:
summary = {
    "pair" : ["MA/V" , "GS/MS" , "Portfolio"],
    "total_return_pct" : [
        round((pnl_MA_V.sum() / 10000) * 100 , 2),
        round((pnl_GS_MS.sum()/10000) * 100 , 2),
        round((portfolio_pnl.sum() / 20000) * 100 ,2)

    ], 
    "sharpe" :[
        sharpe_ratio(pnl_MA_V),
        sharpe_ratio(pnl_GS_MS),
        sharpe_ratio(portfolio_pnl)
    ] , 
    "max_drawdown_pct" : [
        max_drawdown(pnl_MA_V),
        max_drawdown(pnl_GS_MS),
        max_drawdown(portfolio_pnl)
    ]
}

summary_df = pd.DataFrame(summary)
summary_df.to_csv("../data/baseline_metrics.csv", index = False)
print(summary_df)

        pair  total_return_pct  sharpe  max_drawdown_pct
0       MA/V             10.19  0.4140             -7.50
1      GS/MS             12.05  0.3431             -7.96
2  Portfolio             11.12  0.5225             -8.55
